In [15]:
import pandas as pd

# 데이터 불러오기
df=pd.read_csv('https://bit.ly/fish_csv_data')
df

# df.to_excel("fish.xlsx",index=False)

df.info()
df.describe()

<class 'pandas.DataFrame'>
RangeIndex: 159 entries, 0 to 158
Data columns (total 6 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Species   159 non-null    str    
 1   Weight    159 non-null    float64
 2   Length    159 non-null    float64
 3   Diagonal  159 non-null    float64
 4   Height    159 non-null    float64
 5   Width     159 non-null    float64
dtypes: float64(5), str(1)
memory usage: 7.6 KB


,Weight,Length,Diagonal,Height,Width
count,159.000000,159.000000,159.000000,159.000000,159.000000
mean,398.326415,28.415723,31.227044,8.970994,4.417486
std,357.978317,10.716328,11.610246,4.286208,1.685804
min,0.000000,8.400000,8.800000,1.728400,1.047600
25%,120.000000,21.000000,23.150000,5.944800,3.385650
50%,273.000000,27.300000,29.400000,7.786000,4.248500
75%,650.000000,35.500000,39.650000,12.365900,5.584500
max,1650.000000,63.400000,68.000000,18.957000,8.142000


In [16]:
corr = df.corr(numeric_only=True)
print(corr)

# 직관적 해석
# Weight ↑ => 큰 물고기
# Length / Diagonal (대각선) ↑ => 길쭉한 물고기
# Height / Width ↑ => 두꺼운 물고기


            Weight    Length  Diagonal    Height     Width
Weight    1.000000  0.918618  0.923044  0.724345  0.886507
Length    0.918618  1.000000  0.994103  0.640441  0.873547
Diagonal  0.923044  0.994103  1.000000  0.703409  0.878520
Height    0.724345  0.640441  0.703409  1.000000  0.792881
Width     0.886507  0.873547  0.878520  0.792881  1.000000


In [17]:
# 전처리

from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
df["Species"] = le.fit_transform(df["Species"])

df

,Species,Weight,Length,Diagonal,Height,Width
0,0,242.0,25.4,30.0,11.5200,4.0200
1,0,290.0,26.3,31.2,12.4800,4.3056
2,0,340.0,26.5,31.1,12.3778,4.6961
3,0,363.0,29.0,33.5,12.7300,4.4555
4,0,430.0,29.0,34.0,12.4440,5.1340
...,...,...,...,...,...,...
154,5,12.2,12.2,13.4,2.0904,1.3936
155,5,13.4,12.4,13.5,2.4300,1.2690
156,5,12.2,13.0,13.8,2.2770,1.2558
157,5,19.7,14.3,15.2,2.8728,2.0672


In [18]:
# x = df.drop(["Species"],axis=1)
# y = df['Species']

# 다중 공선성 제거의 목적은 “예측률 향상”이 아니라 “모델 안정성과 해석 가능성 향상”

# Weight ↑
# Length ↑
# Diagonal ↑

# 다중 공선성 제거를 위해 위의 3개의 컬럼중 두개를 제거 = 서로 거의 같은 정보.

x = df[['Length', 'Height', 'Width']]
y = df["Species"]

In [19]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

X_train, X_test, y_train, y_test = train_test_split(
    x, y, test_size=0.2, random_state=42
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# scaler로 정규화 시킬 때 처음에 fit_transform 을 사용하는데 그 다음 test에서는 이미 해당 데이터에 이미 사용이 되었기 때문에
# 중복해서 사용하지 않고 transform 이라고만 하면 됨

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

pred = model.predict(X_test)

In [20]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# average='macro' = 클래스 갯수와 상관 없이 모든 클래스를 똑같이 중요하게 보고 평균내는 방식

print("Accuracy:", accuracy_score(y_test, pred))
print("Precision:", precision_score(y_test, pred, average='macro'))
print("Recall:", recall_score(y_test, pred, average='macro'))
print("F1:", f1_score(y_test, pred, average='macro'))

Accuracy: 0.875
Precision: 0.6703296703296704
Recall: 0.7142857142857143
F1: 0.6883116883116883


c:\miniconda3\envs\env_ds\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [21]:
import pandas as pd

importance = pd.Series(model.coef_[0], index=x.columns)
importance = importance.sort_values(key=abs, ascending=False)

print(importance)

Height    3.087246
Width    -0.358279
Length   -0.104175
dtype: float64
